# Linear System Test of the Trajectory Manifold Learn Controller

In [1]:
import torch
import numpy as np

device = torch.device("cuda")
W = None
x_dim = 4
u_dim = 1
H = 10
spectral_radius = 0.9
seed=234

x0 = np.zeros(x_dim)
x_ref = np.zeros(x_dim)
u_ref = np.zeros(u_dim)

Q = torch.eye(x_dim, device=device)
R = torch.eye(u_dim, device=device)
x_ref_t = torch.tensor(x_ref, dtype=torch.float32, device=device)
u_ref_t = torch.tensor(u_ref, dtype=torch.float32, device=device)

w_dim = (H + 1) * x_dim + H * u_dim



## Sample Controllable System

In [2]:
from src.linear_system import sample_controllable_linear_system, is_controllable

A, B = sample_controllable_linear_system(
        x_dim,
        u_dim,
        spectral_radius=spectral_radius,
        seed=seed,
)
rank, controllable = is_controllable(A, B)
print(f"  controllability rank={rank}, full={controllable}")

  controllability rank=4, full=True


## Generate System Trajectories

In [1]:
import importlib
import src.linear_system
importlib.reload(src.linear_system)

<module 'src.linear_system' from 'c:\\Users\\bayer\\PycharmProjects\\koopman\\src\\linear_system.py'>

In [3]:
num_steps = 100
n_repeats = 100
process_noise_std = 0.0

In [4]:
from src.linear_system import generate_linear_trajectory_data
from src.manifold_control import build_trajectory_training_matrix

_, X_all, U_all = generate_linear_trajectory_data(
    A,
    B,
    num_steps=num_steps,
    n_repeats=n_repeats,
    process_noise_std=process_noise_std,
    seed=seed,
)

W = build_trajectory_training_matrix(
    X_all,
    U_all,
    horizon=H,
    device=device,
    dtype=torch.float32,
)
w_dim = W.shape[1]
print(f"  training matrix shape={tuple(W.shape)}")

  training matrix shape=(9100, 54)


# Train Manifold Decoder

In [11]:
import importlib
import src.manifold_control
importlib.reload(src.manifold_control)

<module 'src.manifold_control' from 'c:\\Users\\bayer\\PycharmProjects\\koopman\\src\\manifold_control.py'>

In [7]:
from pathlib import Path

alpha_dim = 16
hidden_dims = [64, 64]
epochs = 100
max_iter = 1000
train_lr = 1e-3

print_every = 10
checkpoint = Path("saves") / "saved_models" / "behavior_decoder_linear.pt"

In [8]:
from src.manifold_control import train_decoder

decoder = train_decoder(
    W,
    x_dim=x_dim,
    u_dim=u_dim,
    horizon=H,
    alpha_dim=alpha_dim,
    hidden_dims=hidden_dims,
    epochs=epochs,
    max_iter=max_iter,
    lr=train_lr,
    print_every=print_every,
    checkpoint=checkpoint,
    device=device,
)
decoder.eval()

Training W=(9100, 54), alpha_table=(9100, 16), device=cuda


epoch=0000, loss=0.511377, fit=0.511377
epoch=0010, loss=0.315971, fit=0.315971
epoch=0020, loss=0.154200, fit=0.154200
epoch=0030, loss=0.087920, fit=0.087920
epoch=0040, loss=0.043098, fit=0.043098
epoch=0050, loss=0.008592, fit=0.008592
epoch=0060, loss=0.004401, fit=0.004401
epoch=0070, loss=0.003854, fit=0.003854
epoch=0080, loss=0.003748, fit=0.003748
epoch=0090, loss=0.003739, fit=0.003739
saved decoder checkpoint to saves\saved_models\behavior_decoder_linear.pt


BehaviorDecoder(
  (net): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=54, bias=True)
  )
)

## Load the trained model

In [ ]:
from src.manifold_control import load_decoder

decoder = load_decoder(
    checkpoint=checkpoint,
    alpha_dim=alpha_dim,
    w_dim=w_dim,
    hidden_dims=hidden_dims,
    device=device,
)

# Solve the Control problem

In [ ]:
x_ref = np.zeros(x_dim)
u_ref = np.zeros(u_dim)
lambda_theta = 1.0
lambda_curvature = 1.0
solve_lr = 1e-3
u_max = 100.0

## Single Step Control

In [10]:
from src.manifold_control import BehaviorManifoldControlSolver


solver = BehaviorManifoldControlSolver(
        decoder=decoder,
        x_dim=x_dim,
        u_dim=u_dim,
        horizon=H,
        Q=Q,
        R=R,
        x_ref=x_ref,
        u_ref=u_ref,
        lambda_theta=lambda_theta,
        lambda_curvature=lambda_curvature,
        lr=solve_lr,
        max_iter=max_iter,
        u_bounds=(-u_max, u_max),
        curvature_mode="local",
        device=device,
    )

x_init = torch.zeros(H + 1, x_dim, device=device)
x_init[0] = torch.tensor(x0, dtype=torch.float32, device=device)
u_init = torch.zeros(H, u_dim, device=device)
alpha_init = torch.zeros(alpha_dim, device=device)

solution = solver.solve(
    x_init=x_init,
    u_init=u_init,
    alpha_init=alpha_init,
    freeze={"theta": True, "x": False, "u": False, "alpha": False},
)

u_plan = solution.u.detach().cpu().numpy()
print(f"  loss_dict={solution.loss_dict}")
print(f"  first control={u_plan[0]}")

  loss_dict={'qr': 8.975814580480801e-08, 'fit': 4.842753241973696e-06, 'curvature': 1.2644492564106713e-08, 'total': 4.945155978930416e-06}
  first control=[4.4668623e-06]


## Closed Loop Control

In [15]:
import importlib
import src.linear_system
importlib.reload(src.linear_system)

<module 'src.linear_system' from 'c:\\Users\\bayer\\PycharmProjects\\koopman\\src\\linear_system.py'>

In [ ]:
sim_steps = 25
x_ref_t = torch.tensor(x_ref, dtype=torch.float32, device=device)
u_ref_t = torch.tensor(u_ref, dtype=torch.float32, device=device)
results_path = Path("saves") / "simulation_results" / "linear_manifold_results.npz"
plot_path = Path("saves") / "figures" / "linear_manifold_results.png"

In [ ]:
from src.linear_system import simulate_discrete_closed_loop, finite_horizon_lqr_u_caller
from src.manifold_control import make_linear_manifold_u_caller

u_manifold = make_linear_manifold_u_caller(
    decoder=decoder,
    Q=Q,
    R=R,
    x_ref=x_ref_t,
    u_ref=u_ref_t,
    H=H,
    alpha_dim=alpha_dim,
    umax=u_max,
    device=device,
    solve_lr=solve_lr,
    max_iter=max_iter,
    lambda_theta=lambda_theta,
    lambda_curvature=lambda_curvature,
)

X_mani, U_mani = simulate_discrete_closed_loop(
    A,
    B,
    u_manifold,
    x0,
    num_steps=sim_steps,
)

u_lqr = finite_horizon_lqr_u_caller(
    A,
    B,
    Q,
    R,
    horizon=max(sim_steps, H),
    x_ref=x_ref,
    umax=u_max,
)
X_lqr, U_lqr = simulate_discrete_closed_loop(
    A,
    B,
    u_lqr,
    x0,
    num_steps=sim_steps,
)

zero_u = lambda _k, _x: np.zeros(u_dim)
X_zero, U_zero = simulate_discrete_closed_loop(
    A,
    B,
    zero_u,
    x0,
    num_steps=sim_steps,
)

print(f"  final ||x_manifold - x_ref||={np.linalg.norm(X_mani[:, -1] - x_ref):.6f}")
print(f"  final ||x_lqr      - x_ref||={np.linalg.norm(X_lqr[:, -1] - x_ref):.6f}")
print(f"  final ||x_zero     - x_ref||={np.linalg.norm(X_zero[:, -1] - x_ref):.6f}")

Simulation Steps:  44%|████▍     | 22/50 [04:23<05:28, 11.73s/it]

## Save Results

In [ ]:
results_path.parent.mkdir(parents=True, exist_ok=True)
np.savez(
    results_path,
    A=A,
    B=B,
    x0=x0,
    x_ref=x_ref,
    X_manifold=X_mani,
    U_manifold=U_mani,
    X_lqr=X_lqr,
    U_lqr=U_lqr,
    X_zero=X_zero,
    U_zero=U_zero,
)

## Load Results

## Plot Results